# 1.0 — Setup & Imports

Import all required standard, scientific, and deep-learning libraries.


In [1]:
# ── Kaggle environment setup ──────────────────────────────────────────────────
import os
import matplotlib
matplotlib.use('Agg')  # headless – no display on Kaggle

import tensorflow as tf

# Allow GPU memory growth (avoids OOM on shared Kaggle GPU)
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print('GPU devices:', gpus)
print('TF version :', tf.__version__)

GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
TF version : 2.21.0


In [2]:
# ── Standard library ────────────────────────────────────────────────────────────
import glob
import json
import time
import random
from pathlib import Path

from scipy.io import loadmat

# ── Scientific / data ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ──────────────────────────────────────────────────────────────
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# ── Deep learning ──────────────────────────────────────────────────────────────
from tensorflow import keras
from tensorflow.keras import layers

# ── ML utilities ───────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, confusion_matrix,
    classification_report, f1_score, log_loss
)

# ── Excel export ───────────────────────────────────────────────────────────────
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.drawing.image import Image as XLImage

sns.set_style('whitegrid')
print('TensorFlow:', tf.__version__)

TensorFlow: 2.21.0


# 2.0 — Configuration

All paths, seeds, dataset geometry, training hyperparameters, and per-architecture configs in one place. **Edit only this section between experiments.**


In [3]:
# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_KEY  = "indian_pines"
LABEL_KEY = "indian_pines_gt"
DATASET_NAME = "Indian Pines"

# ── Directory layout (Kaggle or Local) ─────────────────────────────────────────
_data_matches = (glob.glob("/kaggle/kin/**/Indian_pines.mat", recursive=True) or glob.glob("/kaggle/kin/**/k001.mat", recursive=True))
_ref_matches  = (glob.glob("/kaggle/kin/**/Indian_pines_gt.mat", recursive=True) or glob.glob("/kaggle/kin/**/k002.mat", recursive=True))

if _data_matches and _ref_matches:
    DATA_FILE  = Path(_data_matches[0])
    LABEL_FILE = Path(_ref_matches[0])
    DATA_DIR   = DATA_FILE.parent
else:
    local_candidates = [
        Path("data"),
        Path("examples/indian_pines_uncertainty_quantification/data"),
        Path("../data"),
        Path("/kaggle/kin/indian-pines-dataset")
    ]
    DATA_DIR = next((p for p in local_candidates if (p / "Indian_pines.mat").exists()), Path("data"))
    DATA_FILE = DATA_DIR / "Indian_pines.mat"
    LABEL_FILE = DATA_DIR / "Indian_pines_gt.mat"

# Ensemble teacher models directory
ens_candidates = (
    glob.glob("/kaggle/working/models/**/*_ens_*_final.keras", recursive=True)
    or glob.glob("/kaggle/kin/indian-pines-dataset-models/**/*_ens_*_final.keras", recursive=True)
    or glob.glob("/kaggle/kin/**/*_ens_*_final.keras", recursive=True)
    or glob.glob("models/**/*_ens_*_final.keras", recursive=True)
)
ENSEMBLE_DIR = Path(ens_candidates[0]).parent if ens_candidates else Path("models")

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle").exists() else Path(".")
MODEL_DIR   = WORKING_DIR / "models"
RESULTS_DIR = WORKING_DIR / "results"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Geometry & Training Hyperparameters ────────────────────────────────────────
H, W, B     = 145, 145, 220
PERCENTILE_CLIP = (2.0, 98.0)
PATCH_SIZE  = 9
INNER_PATCH = 3
NUM_CLASSES = 16

TRAIN_PERCENT = 0.75
VAL_PERCENT   = 0.10
TEST_PERCENT  = 0.15

BATCH_SIZE    = 128
STUDENT_EPOCHS = 100
PATIENCE      = 15
LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 1e-4

ALPHA_BALANCING = 0.5
EPS             = 1e-8

print(f"Data file:     {DATA_FILE}")
print(f"Label file:    {LABEL_FILE}")
print(f"Ensemble dir:  {ENSEMBLE_DIR}")
print(f"Model dir:     {MODEL_DIR}")
print(f"Results dir:   {RESULTS_DIR}")

# ── Restored CREDIT Hyperparameters (auto-restored) ──
PLOT_DIR = RESULTS_DIR / 'training_plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)
VAL_SPLIT_FROM_TRAIN = 0.20
EPOCHS = 100
DROPOUT_RATE = 0.25
TRAIN_CFG = {
    'label_smoothing': 0.05,
    'weight_decay': 1e-4,
    'clipnorm': 1.0,
    'cosine_alpha': 0.05,
}
ALEXNET_CFG = {
    'conv_filters': [96, 256, 384, 384, 256],
    'dense_units': [4096, 1024, 256, 32],
}
GFNET_CFG = {'hidden_dim': 512, 'num_blocks': 5, 'mlp_ratio': 4}
VIT_CFG = {
    'projection_dim': 256,
    'num_heads': 4,
    'transformer_layers': 12,
    'mlp_multiplier': 2,
    'head_units': [512, 256, 128, 64],
}
ALEXNET_LEGACY_SPLIT_SEED = 10
ALEXNET_LEGACY_TRAIN_PERCENT = 0.75
ALEXNET_LR_START = 0.01
ALEXNET_LR_MAX = 0.02
ALEXNET_LR_MIN = 0.005


Data file:     /kaggle/kin/datasets/daneshselwal/ip-dataset-k01/Indian_pines.mat
Label file:    /kaggle/kin/datasets/daneshselwal/ip-dataset-k01/Indian_pines_gt.mat
Ensemble dir:  /kaggle/working/models/ensembles
Model dir:     /kaggle/working/models
Results dir:   /kaggle/working/results


# 3.0 — Data Loading & Patch Extraction

Reads the hyperspectral/multispectral scene from .mat files, applies a robust per-band percentile-clip normalisation, extracts fixed-size spatial patches centred on every labelled pixel, and builds both the standard and AlexNet-legacy train/test splits.


## 3.1 — Helper Functions


In [4]:
def load_scene_from_mat(data_path, label_path, data_key=DATA_KEY, label_key=LABEL_KEY):
    """Load hyperspectral scene from .mat files (Indian Pines: 145x145x220)."""
    data_mat  = loadmat(data_path)
    label_mat = loadmat(label_path)

    def _first_array(mat_dict, key):
        if key is not None and key in mat_dict:
            return mat_dict[key]
        candidates = [v for k, v in mat_dict.items() if not k.startswith("__")]
        if not candidates:
            raise ValueError(f"No data arrays found in {mat_dict.keys()}")
        return candidates[0]

    x = _first_array(data_mat, data_key).astype(np.float32)
    y = _first_array(label_mat, label_key).astype(np.int32)

    if x.ndim != 3:
        raise ValueError(f"Expected a (H, W, Bands) cube, got shape {x.shape}")
    if y.shape != x.shape[:2]:
        raise ValueError(f"Label shape {y.shape} does not match image shape {x.shape[:2]}")

    height, width, bands = x.shape

    x_norm = np.empty_like(x, dtype=np.float32)
    for b in range(bands):
        band = x[:, :, b]
        lo   = np.percentile(band, PERCENTILE_CLIP[0])
        hi   = np.percentile(band, PERCENTILE_CLIP[1])
        denom = max(hi - lo, 1e-8)
        band_clipped = np.clip(band, lo, hi)
        x_norm[:, :, b] = (band_clipped - lo) / denom

    return x_norm, y, height, width, bands


def extract_labeled_patches(x, y, patch_size=9):
    """Extract spatial patches around every labeled pixel."""
    pad   = patch_size // 2
    x_pad = np.pad(x, ((pad, pad), (pad, pad), (0, 0)), mode="edge")

    coords  = np.argwhere(y > 0)
    patches = np.empty((coords.shape[0], patch_size, patch_size, x.shape[-1]), dtype=np.float32)
    labels  = np.empty((coords.shape[0],), dtype=np.int32)

    for i, (r, c) in enumerate(coords):
        patches[i] = x_pad[r:r + patch_size, c:c + patch_size, :]
        labels[i]  = int(y[r, c]) - 1

    return patches, labels, coords


## 3.2 — Load Image & Extract Patches


In [5]:
print('Loading multispectral image...')
x_img, y_img, H, W, B = load_scene_from_mat(DATA_FILE, LABEL_FILE, data_key=DATA_KEY, label_key=LABEL_KEY)

print('Extracting patches...')
X, y, coords = extract_labeled_patches(x_img, y_img, PATCH_SIZE)

num_classes = int(np.unique(y).size)
input_shape = (PATCH_SIZE, PATCH_SIZE, B)

print(f'Scene geometry: H={H}  W={W}  Bands={B}')
print(f'Image  : {x_img.shape}  |  Labels : {y_img.shape}')
print(f'Patches: {X.shape}  |  Classes: {num_classes}')


Loading multispectral image...


Extracting patches...


Scene geometry: H=145  W=145  Bands=220
Image  : (145, 145, 220)  |  Labels : (145, 145)
Patches: (10249, 9, 9, 220)  |  Classes: 16


## 3.3 — Train / Validation / Test Split

Two independent splits are created: a **standard split** (shared by GFNet and ViT) and an **AlexNet legacy split** (separate seed to preserve reproducibility of uncertainty recovery).


In [6]:
# [auto-fix: hoisted definition so it exists before first use]  <VAL_SPLIT_FROM_TRAIN>
VAL_SPLIT_FROM_TRAIN = 0.20

# FIX H13: the silent override of the AlexNet split (20 % / seed 42) is removed;
# every model now uses the shared split below.


In [7]:
# ── Standard split (GFNet & ViT) ──────────────────────────────────────────────
# FIX C1/C3/C4 — shared train / val / cal / eval split (identical code in every notebook) ──
# SPLIT_MODE = "spatial_block": whole spatial blocks go to one role and training
#   pixels whose 9x9 patch would contain a cal/eval pixel are dropped, so no
#   labelled test pixel is ever seen inside a training patch.
# SPLIT_MODE = "random": legacy stratified pixel-wise split (spatially leaky).
import numpy as _np_split
SPLIT_MODE        = globals().get("SPLIT_MODE", "spatial_block")
SPLIT_SEED        = 42
SPLIT_BLOCK       = 10
SPLIT_TRAIN_FRAC  = 0.75   # fraction of labelled pixels assigned to train+val
SPLIT_VAL_FRAC    = 0.20   # fraction of train+val used for validation / checkpointing
SPLIT_CAL_FRAC    = 0.50   # fraction of the held-out pool used for conformal calibration


def make_shared_split(y_img, patch_size=9, mode=None, seed=SPLIT_SEED,
                      train_frac=SPLIT_TRAIN_FRAC, val_frac=SPLIT_VAL_FRAC,
                      cal_frac=SPLIT_CAL_FRAC, block=SPLIT_BLOCK):
    """Return dict of index arrays (into np.argwhere(y_img > 0) order):
    train, val, cal, eval, test (= cal U eval)."""
    mode   = mode or SPLIT_MODE
    coords = _np_split.argwhere(y_img > 0)
    labels = y_img[y_img > 0].astype(int) - 1
    n, K   = len(labels), int(labels.max()) + 1
    rng    = _np_split.random.default_rng(seed)
    roles  = _np_split.full(n, -1)                       # 0 train, 1 val, 2 cal, 3 eval
    held   = 1.0 - train_frac
    frac   = _np_split.array([train_frac * (1 - val_frac), train_frac * val_frac,
                              held * cal_frac, held * (1 - cal_frac)])
    if mode == "random":
        from sklearn.model_selection import train_test_split as _tts
        idx = _np_split.arange(n)
        def _strat(a):
            cnt = _np_split.bincount(labels[a], minlength=K)
            return labels[a] if cnt[cnt > 0].min() >= 2 else None
        tv, te = _tts(idx, train_size=train_frac, random_state=seed, stratify=_strat(idx))
        tr, va = _tts(tv, test_size=val_frac, random_state=seed, stratify=_strat(tv))
        ca, ev = _tts(te, test_size=1 - cal_frac, random_state=seed, stratify=_strat(te))
        for r, a in enumerate([tr, va, ca, ev]):
            roles[a] = r
    else:
        bid = (coords[:, 0] // block) * 100000 + (coords[:, 1] // block)
        ub, inv = _np_split.unique(bid, return_inverse=True)
        cnt = _np_split.zeros((len(ub), K))
        _np_split.add.at(cnt, (inv, labels), 1)
        target = frac[None, :] * cnt.sum(0)[:, None]          # (K, 4)
        have   = _np_split.zeros((K, 4))
        order  = rng.permutation(len(ub))
        # rare classes first so each role receives some of them
        rarity = (cnt / _np_split.maximum(cnt.sum(0, keepdims=True), 1)).max(1)
        order  = order[_np_split.argsort(-rarity[order], kind="stable")]
        for b in order:
            w    = cnt[b]
            need = ((target - have) / _np_split.maximum(target, 1)) * w[:, None]
            role = int(_np_split.argmax(need.sum(0)))
            have[:, role] += w
            roles[inv == b] = role
        # drop training/validation pixels whose patch would contain a held-out pixel
        pad  = patch_size // 2
        grid = _np_split.zeros(y_img.shape, dtype=bool)
        hc   = coords[roles >= 2]
        grid[hc[:, 0], hc[:, 1]] = True
        from scipy.ndimage import maximum_filter as _mf
        near = _mf(grid.astype(_np_split.uint8), size=2 * pad + 1) > 0
        drop = (roles <= 1) & near[coords[:, 0], coords[:, 1]]
        roles[drop] = -1
    out = {k: _np_split.where(roles == r)[0] for r, k in enumerate(["train", "val", "cal", "eval"])}
    out["test"] = _np_split.sort(_np_split.concatenate([out["cal"], out["eval"]]))
    out["coords"], out["labels"], out["mode"] = coords, labels, mode
    return out


def check_split_alignment(split, ns):
    """Fail loudly if the notebook's labelled-patch order differs from the shared split order
    (row-major over y > 0). Any 1-D label vector of the same length is compared up to relabelling."""
    lab = split["labels"]; n = len(lab); checked = []
    for name in ("y", "Y", "y_all", "y_labels", "y_labeled"):
        v = ns.get(name)
        if isinstance(v, _np_split.ndarray) and v.ndim == 1 and len(v) == n:
            pairs = set(zip(lab.tolist(), _np_split.asarray(v).astype(int).tolist()))
            if len(pairs) != len(set(lab.tolist())) or len({b for _, b in pairs}) != len(pairs):
                raise RuntimeError(f"shared split is not aligned with label vector '{name}'")
            checked.append(name)
    print("split alignment verified against:", checked or "nothing (no 1-D label vector in scope)")


def describe_split(split, num_classes):
    import pandas as _pd
    rows = {k: _np_split.bincount(split["labels"][split[k]], minlength=num_classes)
            for k in ["train", "val", "cal", "eval"]}
    df = _pd.DataFrame(rows); df.index.name = "class"
    print(f"Split mode: {split['mode']}  |  sizes:", {k: int(v.sum()) for k, v in rows.items()})
    return df


SPLIT = make_shared_split(y_img, patch_size=PATCH_SIZE)
check_split_alignment(SPLIT, globals())
display(describe_split(SPLIT, num_classes))
x_train, y_train = X[SPLIT["train"]], y[SPLIT["train"]]
x_val,   y_val   = X[SPLIT["val"]],   y[SPLIT["val"]]
x_test,  y_test  = X[SPLIT["test"]],  y[SPLIT["test"]]

y_train_cat = keras.utils.to_categorical(y_train, num_classes)
y_val_cat   = keras.utils.to_categorical(y_val,   num_classes)
y_test_cat  = keras.utils.to_categorical(y_test,  num_classes)

print(f'Train : {x_train.shape}  Val : {x_val.shape}  Test : {x_test.shape}')

# ── AlexNet legacy split (separate seed for uncertainty recovery) ──────────────
# FIX C1/H13: AlexNet shares the split of the other backbones
x_train_alex, x_test_alex, y_train_alex, y_test_alex = x_train, x_test, y_train, y_test
y_train_alex_cat = keras.utils.to_categorical(y_train_alex, num_classes)
y_test_alex_cat  = keras.utils.to_categorical(y_test_alex,  num_classes)

print(f'AlexNet  Train : {x_train_alex.shape}  Test : {x_test_alex.shape}')

split alignment verified against: ['y']
Split mode: spatial_block  |  sizes: {'train': 3042, 'val': 985, 'cal': 1426, 'eval': 1385}


,train,val,cal,eval
class,,,,
0,0,3,15,0
1,403,152,197,188
2,208,40,110,111
3,79,11,31,54
4,168,54,72,77
5,132,90,106,91
6,14,14,0,0
7,109,60,90,90
8,20,0,0,0


Train : (3042, 9, 9, 220)  Val : (985, 9, 9, 220)  Test : (2811, 9, 9, 220)
AlexNet  Train : (3042, 9, 9, 220)  Test : (2811, 9, 9, 220)


# 4.0 — Model Definitions

Defines all three single-head base architectures (AlexNet CNN, GFNet, ViT with U-Net skip connections) plus the shared custom Keras layers used by GFNet and ViT.


## 4.1 — Shared Custom Keras Layers

Four serialisable custom layers: patch extraction, positional encoding (with and without CLS token), and the frequency-domain Global Filter layer used in GFNet.


In [8]:
@tf.keras.utils.register_keras_serializable()
class PatchExtractor(layers.Layer):
    """Extract non-overlapping patches from a spatial image tensor."""

    def __init__(self, patch_size=3, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding='VALID',
        )
        batch       = tf.shape(images)[0]
        num_patches = tf.shape(patches)[1] * tf.shape(patches)[2]
        patch_dim   = tf.shape(patches)[-1]
        return tf.reshape(patches, [batch, num_patches, patch_dim])

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'patch_size': self.patch_size})
        return cfg


@tf.keras.utils.register_keras_serializable()
class PatchPositionEncoder(layers.Layer):
    """Linear projection + learnable position embedding (no CLS token)."""

    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches    = num_patches
        self.projection_dim = projection_dim
        self.projection         = layers.Dense(projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches, output_dim=projection_dim
        )

    def call(self, patches):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        return self.projection(patches) + self.position_embedding(positions)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'num_patches': self.num_patches, 'projection_dim': self.projection_dim})
        return cfg


@tf.keras.utils.register_keras_serializable()
class PatchEncoderWithCLS(layers.Layer):
    """Linear projection + learnable position embedding WITH a prepended CLS token."""

    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches    = num_patches
        self.projection_dim = projection_dim
        self.projection         = layers.Dense(projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches + 1, output_dim=projection_dim
        )

    def build(self, input_shape):
        self.cls_token = self.add_weight(
            name='cls_token', shape=(1, 1, self.projection_dim),
            initializer='zeros', trainable=True
        )
        super().build(input_shape)

    def call(self, patches):
        batch      = tf.shape(patches)[0]
        patch_proj = self.projection(patches)
        cls_tokens = tf.repeat(self.cls_token, repeats=batch, axis=0)
        x          = tf.concat([cls_tokens, patch_proj], axis=1)
        positions  = tf.range(start=0, limit=self.num_patches + 1, delta=1)
        return x + self.position_embedding(positions)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'num_patches': self.num_patches, 'projection_dim': self.projection_dim})
        return cfg


@tf.keras.utils.register_keras_serializable()
class GlobalFilterLayer(layers.Layer):
    """Learnable complex-valued filter applied in the 2-D frequency domain."""

    def __init__(self, token_side, **kwargs):
        super().__init__(**kwargs)
        self.token_side = token_side

    def build(self, input_shape):
        channels = int(input_shape[-1])
        self.w_real = self.add_weight(
            name='w_real', shape=(self.token_side, self.token_side, channels),
            initializer='glorot_uniform', trainable=True
        )
        self.w_imag = self.add_weight(
            name='w_imag', shape=(self.token_side, self.token_side, channels),
            initializer='zeros', trainable=True
        )
        super().build(input_shape)

    def call(self, x):
        batch    = tf.shape(x)[0]
        channels = tf.shape(x)[-1]
        x_2d       = tf.reshape(x, [batch, self.token_side, self.token_side, channels])
        # FIX C5: fft2d transforms the two innermost axes, so move channels first and
        # filter over the (token_row, token_col) spatial grid as GFNet intends.
        x_2d       = tf.transpose(x_2d, [0, 3, 1, 2])                       # (B, C, s, s)
        x_fft      = tf.signal.fft2d(tf.cast(x_2d, tf.complex64))
        w_complex  = tf.transpose(tf.complex(self.w_real, self.w_imag), [2, 0, 1])
        x_filtered = x_fft * w_complex
        x_spatial  = tf.math.real(tf.signal.ifft2d(x_filtered))
        x_spatial  = tf.transpose(x_spatial, [0, 2, 3, 1])                  # (B, s, s, C)
        return tf.reshape(x_spatial, [batch, self.token_side * self.token_side, channels])

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'token_side': self.token_side})
        return cfg

## 4.2 — AlexNet CNN


In [9]:
def build_alexnet(input_shape, num_classes, dropout_rate=0.25, cfg=None):
    """Build a single-head AlexNet-style CNN with configurable conv and dense units."""
    cfg = cfg or ALEXNET_CFG

    inputs = keras.Input(shape=input_shape)
    x = inputs

    for i, filters in enumerate(cfg['conv_filters'], start=1):
        x = layers.Conv2D(filters, (3, 3), activation='relu',
                          padding='same', name=f'alex_conv_{i}')(x)

    x = layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2),
                            padding='same', name='alex_pool')(x)
    x = layers.Flatten(name='alex_flatten')(x)

    fc_names   = ['alex_fc1', 'alex_fc2', 'alex_fc3', 'alex_fc4']
    drop_names = ['TRAIN_DROPOUT_1', 'TRAIN_DROPOUT_2', 'TRAIN_DROPOUT_3', None]

    for units, fc_name, drop_name in zip(cfg['dense_units'], fc_names, drop_names):
        x = layers.Dense(units, activation='relu', name=fc_name)(x)
        if drop_name:
            x = layers.Dropout(dropout_rate, name=drop_name)(x)

    outputs = layers.Dense(num_classes, activation='softmax', name='alex_logits')(x)
    return keras.Model(inputs, outputs, name='AlexNet_SingleHead')

## 4.3 — Global Filter Network (GFNet)


In [10]:
def gf_block(x, token_side, dim, mlp_ratio=4, dropout_rate=0.25, name_prefix='gf'):
    """One GFNet block: Global Filter → LayerNorm → MLP → residual add."""
    y = layers.LayerNormalization(name=f'{name_prefix}_ln1')(x)
    y = GlobalFilterLayer(token_side, name=f'{name_prefix}_gfilter')(y)
    y = layers.LayerNormalization(name=f'{name_prefix}_ln2')(y)
    y = layers.Dense(dim * mlp_ratio, activation=tf.keras.activations.gelu,
                     name=f'{name_prefix}_mlp1')(y)
    y = layers.Dropout(dropout_rate, name=f'{name_prefix}_drop1')(y)
    y = layers.Dense(dim, activation=tf.keras.activations.gelu,
                     name=f'{name_prefix}_mlp2')(y)
    y = layers.Dropout(dropout_rate, name=f'{name_prefix}_drop2')(y)
    return layers.Add(name=f'{name_prefix}_add')([x, y])


def build_gfnet(input_shape, num_classes, inner_patch=3,
                hidden_dim=512, num_blocks=5, mlp_ratio=4, dropout_rate=0.25):
    """Build a single-head Global Filter Network over tokenised image patches."""
    num_patches = (input_shape[0] // inner_patch) * (input_shape[1] // inner_patch)
    token_side  = int(np.sqrt(num_patches))

    inputs = keras.Input(shape=input_shape)
    x = PatchExtractor(inner_patch, name='gf_patch_extractor')(inputs)
    x = PatchPositionEncoder(num_patches, hidden_dim, name='gf_patch_encoder')(x)
    x = layers.Dropout(dropout_rate, name='TRAIN_DROPOUT_1')(x)

    for i in range(num_blocks):
        x = gf_block(x, token_side, hidden_dim, mlp_ratio,
                     dropout_rate, name_prefix=f'gf_block_{i+1}')

    x = layers.Dropout(dropout_rate, name='TRAIN_DROPOUT_2')(x)
    x = layers.LayerNormalization(name='gf_final_ln')(x)
    x = layers.GlobalAveragePooling1D(name='gf_gap')(x)
    x = layers.Flatten(name='gf_flatten')(x)
    x = layers.Dropout(dropout_rate, name='TRAIN_DROPOUT_3')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='gf_logits')(x)

    return keras.Model(inputs, outputs, name='GFNet_SingleHead')

## 4.4 — Vision Transformer with U-Net Skip Connections


In [11]:
def transformer_block(x, num_heads, projection_dim, mlp_dim, dropout_rate, name_prefix):
    """Standard pre-LN transformer block: Multi-Head Attention + MLP with residual adds."""
    y = layers.LayerNormalization(epsilon=1e-6, name=f'{name_prefix}_ln1')(x)
    y = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=projection_dim,
        dropout=dropout_rate, name=f'{name_prefix}_mha'
    )(y, y)
    x = layers.Add(name=f'{name_prefix}_add1')([y, x])

    y = layers.LayerNormalization(epsilon=1e-6, name=f'{name_prefix}_ln2')(x)
    y = layers.Dense(mlp_dim, activation=tf.keras.activations.gelu,
                     name=f'{name_prefix}_mlp1')(y)
    y = layers.Dropout(dropout_rate, name=f'{name_prefix}_drop1')(y)
    y = layers.Dense(projection_dim, activation=tf.keras.activations.gelu,
                     name=f'{name_prefix}_mlp2')(y)
    y = layers.Dropout(dropout_rate, name=f'{name_prefix}_drop2')(y)
    return layers.Add(name=f'{name_prefix}_add2')([y, x])


def build_vit_unet_singlehead(
    input_shape, num_classes, inner_patch=3,
    projection_dim=256, num_heads=4, transformer_layers=12,
    mlp_multiplier=2, dropout_rate=0.25, head_units=(512, 256, 128, 64)
):
    """Build a single-head ViT with encoder-decoder skip connections (U-Net style)."""
    num_patches = (input_shape[0] // inner_patch) * (input_shape[1] // inner_patch)

    inputs = keras.Input(shape=input_shape)
    x = PatchExtractor(inner_patch, name='vit_patch_extractor')(inputs)
    x = PatchEncoderWithCLS(num_patches, projection_dim, name='vit_patch_encoder')(x)

    block_list = []
    for i in range(transformer_layers):
        x = transformer_block(
            x, num_heads=num_heads, projection_dim=projection_dim,
            mlp_dim=projection_dim * mlp_multiplier,
            dropout_rate=0.1, name_prefix=f'vit_block_{i+1}'
        )
        if i <= transformer_layers // 2:
            block_list.append(x)
        else:
            x = layers.Add(name=f'vit_skip_add_{i+1}')(
                [x, block_list[transformer_layers - i - 1]]
            )

    x         = layers.Dropout(dropout_rate, name='TRAIN_DROPOUT_1')(x)
    x         = layers.LayerNormalization(epsilon=1e-6, name='vit_cls_norm')(x)
    cls_token = layers.Lambda(lambda t: t[:, 0, :], name='vit_cls_token')(x)

    drop_slots = ['TRAIN_DROPOUT_3', None, 'TRAIN_DROPOUT_5', 'TRAIN_DROPOUT_6']
    head_names = ['vit_head_1', 'vit_head_2', 'vit_head_3', 'vit_head_4']
    y = cls_token
    for units, hname, dname in zip(head_units, head_names, drop_slots):
        y = layers.Dense(units, activation=tf.keras.activations.gelu, name=hname)(y)
        if dname:
            y = layers.Dropout(dropout_rate, name=dname)(y)

    outputs = layers.Dense(num_classes, activation='softmax', name='vit_logits')(y)
    return keras.Model(inputs, outputs, name='ViT_UNet_SingleHead')

# 5.0 — Training Helpers

Utility functions for calibration metrics, optimiser construction, and the formatted Excel report writer.


## 5.1 — Calibration Metrics


In [12]:
def multiclass_brier_score(y_onehot, y_prob):
    """Compute the multiclass Brier score (mean squared probability error)."""
    return float(np.mean(np.sum((y_prob - y_onehot) ** 2, axis=1)))


def expected_calibration_error(y_true, y_prob, n_bins=15):
    """Compute Expected Calibration Error (ECE) using equal-width confidence bins."""
    confidences = np.max(y_prob, axis=1)
    predictions = np.argmax(y_prob, axis=1)
    correct     = (predictions == y_true).astype(np.float32)
    bin_edges   = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        in_bin = (confidences >= lo) & (
            confidences <= hi if i == n_bins - 1 else confidences < hi
        )
        prop = np.mean(in_bin)
        if prop > 0:
            ece += np.abs(np.mean(correct[in_bin]) - np.mean(confidences[in_bin])) * prop
    return float(ece)

## 5.5 — Credal Prediction-Set Metrics

Coverage-style metrics for the credal (CREDIT) approach, defined exactly like the conformal-prediction ones: overall empirical coverage, average set size and class-wise empirical coverage. Bounds: `lower = p*`, `upper = min(1, p* + Δp)`.


In [13]:
# ── Credal prediction-set metrics (interval dominance) ─────────────────────────
def credal_set_metrics(lower, upper, y_true, num_classes):
    """
    Turn credal probability bounds into prediction SETS and score them like
    conformal prediction does (overall coverage, average set size, class-wise coverage).

    Set rule (interval dominance): class k stays in C(x) unless another class is
    certainly better, i.e.  k in C(x)  <=>  upper_k(x) >= max_j lower_j(x).
    The argmax-of-lower class is always kept, so no set is ever empty.

    lower, upper : (N, C) credal bounds     y_true : (N,) integer labels 0..C-1
    """
    lower  = np.asarray(lower,  dtype=np.float64)
    upper  = np.maximum(np.asarray(upper, dtype=np.float64), lower)
    y_true = np.asarray(y_true).astype(int)
    n      = len(y_true)

    in_set  = upper >= lower.max(axis=1, keepdims=True)      # (N, C) boolean sets
    covered = in_set[np.arange(n), y_true]                   # true label inside set?

    classwise = np.array([
        covered[y_true == k].mean() if np.any(y_true == k) else np.nan
        for k in range(num_classes)
    ])
    return {
        'n':         int(n),
        'coverage':  float(covered.mean()),                  # overall empirical coverage
        'avg_size':  float(in_set.sum(axis=1).mean()),       # average set size
        'classwise': classwise,                              # class-wise empirical coverage
    }


def credal_frames(records):
    """records: list of (model_name, metrics_dict) -> (overall_df, classwise_df)."""
    overall = pd.DataFrame([{
        'Model':                      m,
        'N_Test_Pixels':              r['n'],
        'Overall_Empirical_Coverage': r['coverage'],
        'Average_Set_Size':           r['avg_size'],
        'Mean_Class_Coverage':        float(np.nanmean(r['classwise'])),
    } for m, r in records])
    classwise = pd.DataFrame([{
        'Model': m,
        **{f'Class_{k + 1}_Coverage': float(v) for k, v in enumerate(r['classwise'])},
    } for m, r in records])
    return overall, classwise


def add_credal_coverage_sheet(wb, overall_df, classwise_df, bounds_note, index=1):
    """Add a 'Credal Coverage' sheet (overall + class-wise tables + method notes)."""
    ws = wb.create_sheet('Credal Coverage', index)
    r = _write_df_to_sheet(ws, overall_df, start_row=1,
                           title='Credal prediction sets — overall empirical coverage & average set size') + 1
    r = _write_df_to_sheet(ws, classwise_df, start_row=r,
                           title='Credal prediction sets — class-wise empirical coverage') + 1
    notes = [
        'How the numbers are computed (same definitions as for conformal prediction):',
        bounds_note,
        'Set rule (interval dominance): class k is in the set  iff  upper_k >= max_j lower_j.',
        'Overall coverage = fraction of labelled test pixels whose true class is inside the set.',
        'Average set size = mean number of classes in the set (1 = confident singleton).',
        'Class-wise coverage (class k) = fraction of test pixels of true class k whose set contains k.',
        'Class_k = reference label k (1-based, same as the reference raster; 0 = unlabelled, excluded).',
        'Coverage here is MEASURED on the test set; unlike (split) conformal prediction it is not calibrated to a target 1-alpha.',
    ]
    for line in notes:
        ws.cell(row=r, column=1, value=line).font = Font(name='Arial', size=10, bold=(line == notes[0]))
        r += 1
    return ws


## 5.2 — Optimiser & LR Schedule


In [14]:
def make_adamw_optimizer(num_train_samples):
    """Build a cosine-decay AdamW optimiser (used for GFNet and ViT)."""
    steps_per_epoch = int(np.ceil(num_train_samples / BATCH_SIZE))
    decay_steps     = max(1, steps_per_epoch * EPOCHS)
    lr_schedule = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=LEARNING_RATE,
        decay_steps=decay_steps,
        alpha=TRAIN_CFG['cosine_alpha'],
    )
    return keras.optimizers.AdamW(
        learning_rate=lr_schedule,
        weight_decay=TRAIN_CFG['weight_decay'],
        clipnorm=TRAIN_CFG['clipnorm'],
    )


def _alexnet_legacy_lr(epoch):
    """Cosine LR schedule oscillating between ALEXNET_LR_MIN and ALEXNET_LR_MAX."""
    if EPOCHS <= 1:
        return ALEXNET_LR_START
    phase        = np.pi * epoch / (EPOCHS - 1)
    cosine_decay = 0.5 * (1.0 + np.cos(phase))
    return float((ALEXNET_LR_MAX - ALEXNET_LR_MIN) * cosine_decay + ALEXNET_LR_MIN)

## 5.3 — Excel Report Helpers

Style constants and helper functions for writing DataFrames and plots into a formatted `.xlsx` workbook.


In [15]:
# ── Style constants ────────────────────────────────────────────────────────────
_HDR_FILL   = PatternFill('solid', start_color='1F4E79')
_ALT_FILL   = PatternFill('solid', start_color='D6E4F0')
_HDR_FONT   = Font(name='Arial', bold=True, color='FFFFFF', size=11)
_BODY_FONT  = Font(name='Arial', size=10)
_TITLE_FONT = Font(name='Arial', bold=True, size=13)
_CENTER     = Alignment(horizontal='center', vertical='center', wrap_text=True)
_LEFT       = Alignment(horizontal='left',   vertical='center')
_THIN_SIDE  = Side(style='thin', color='AAAAAA')
_THIN_BORDER = Border(
    left=_THIN_SIDE, right=_THIN_SIDE, top=_THIN_SIDE, bottom=_THIN_SIDE
)


def _style_header_row(ws, row, col_start, col_end):
    """Apply dark-blue header styling to a worksheet row range."""
    for c in range(col_start, col_end + 1):
        cell = ws.cell(row=row, column=c)
        cell.fill      = _HDR_FILL
        cell.font      = _HDR_FONT
        cell.alignment = _CENTER
        cell.border    = _THIN_BORDER


def _style_data_rows(ws, row_start, row_end, col_start, col_end):
    """Apply alternating-row fill and border to a worksheet data range."""
    for r in range(row_start, row_end + 1):
        fill = _ALT_FILL if r % 2 == 0 else PatternFill()
        for c in range(col_start, col_end + 1):
            cell = ws.cell(row=r, column=c)
            cell.font      = _BODY_FONT
            cell.fill      = fill
            cell.alignment = _LEFT
            cell.border    = _THIN_BORDER


def _write_df_to_sheet(ws, df, start_row=1, start_col=1, title=None):
    """Write a DataFrame into a worksheet with headers, alternating rows, and auto-width columns."""
    r = start_row
    if title:
        cell = ws.cell(row=r, column=start_col, value=title)
        cell.font      = _TITLE_FONT
        cell.alignment = _LEFT
        # Guard: merge_cells requires end_column >= start_column (>= 1)
        _end_col = start_col + len(df.columns) - 1
        if _end_col >= start_col:
            ws.merge_cells(
                start_row=r, start_column=start_col,
                end_row=r, end_column=_end_col
            )
        r += 1

    # Header row
    for j, col_name in enumerate(df.columns, start=start_col):
        ws.cell(row=r, column=j, value=col_name)
    _style_header_row(ws, r, start_col, start_col + len(df.columns) - 1)
    r += 1

    # Data rows
    data_start = r
    for _, row_data in df.iterrows():
        for j, val in enumerate(row_data, start=start_col):
            ws.cell(row=r, column=j,
                    value=round(float(val), 6)
                    if isinstance(val, (float, np.floating)) else val)
        r += 1
    _style_data_rows(ws, data_start, r - 1, start_col, start_col + len(df.columns) - 1)

    # Auto column width
    for j, col_name in enumerate(df.columns, start=start_col):
        max_len = max(
            len(str(col_name)),
            max((len(str(ws.cell(row=row, column=j).value or ''))
                 for row in range(data_start, r)), default=0)
        )
        ws.column_dimensions[get_column_letter(j)].width = min(max_len + 4, 40)

    return r  # next free row

## 5.4 — Excel Report Builder

Assembles the full `CREDIT_Results.xlsx` workbook with four sheets: Training Summary, Evaluation Summary, Confusion Matrices, and Plots.


In [16]:
def create_credit_excel_report(
    results_dir,
    train_summary_df,
    eval_summary_df,
    credit_artifacts_dict,
    plot_paths,           # list of (label_str, image_path_str)
    pixel_uncertainty=None,  # dict from prepare_pixel_uncertainty() per model (optional)
    credal_overall_df=None,  # overall coverage / avg set size per model (optional)
    credal_classwise_df=None # class-wise coverage per model (optional)
):
    """
    Create CREDIT_Results.xlsx with these sheets (+ 'Credal Coverage' when provided):
      Sheet 1 — Training Summary
      Sheet 2 — Evaluation Summary
      Sheet 3 — Confusion Matrices (per model, side by side)
      Sheet 4 — Plots (one image per row)
    """
    xlsx_path = Path(results_dir) / 'CREDIT_Results.xlsx'
    wb = Workbook()

    # ── Sheet 1: Training Summary ──────────────────────────────────────────────
    ws1 = wb.active
    ws1.title = 'Training Summary'
    ws1.row_dimensions[1].height = 30
    _write_df_to_sheet(ws1, train_summary_df,
                       start_row=1, title='CREDIT — Training Uncertainty Summary')

    # ── Sheet 2: Evaluation Summary ────────────────────────────────────────────
    ws2 = wb.create_sheet('Evaluation Summary')
    _write_df_to_sheet(ws2, eval_summary_df,
                       start_row=1, title='CREDIT — Full Evaluation Metrics')

    # ── Credal coverage sheet (overall coverage, avg set size, class-wise coverage) ──
    if credal_overall_df is not None and credal_classwise_df is not None:
        add_credal_coverage_sheet(
            wb, credal_overall_df, credal_classwise_df,
            bounds_note='Credal bounds per pixel: lower = p_low (student head, target = min over members), upper = min(1, p_low + delta_p).',
            index=2)

    # ── Sheet 3: Confusion Matrices ────────────────────────────────────────────
    ws3 = wb.create_sheet('Confusion Matrices')
    col_cursor = 1
    for mname, art in credit_artifacts_dict.items():
        cm    = art['confusion_matrix']
        rep   = art['report']
        n_cls = cm.shape[0]

        # Title
        cell = ws3.cell(row=1, column=col_cursor,
                        value=f'{mname} — Confusion Matrix')
        cell.font = _TITLE_FONT
        ws3.merge_cells(start_row=1, start_column=col_cursor,
                        end_row=1, end_column=col_cursor + n_cls)

        # Column headers (Predicted)
        ws3.cell(row=2, column=col_cursor,
                 value='True / Pred').font = Font(bold=True, name='Arial')
        for j in range(n_cls):
            c = ws3.cell(row=2, column=col_cursor + 1 + j, value=f'C{j+1}')
            c.font      = Font(bold=True, name='Arial', color='FFFFFF')
            c.fill      = _HDR_FILL
            c.alignment = _CENTER

        # Data rows
        for i in range(n_cls):
            ws3.cell(row=3 + i, column=col_cursor,
                     value=f'C{i+1}').font = Font(bold=True, name='Arial')
            for j in range(n_cls):
                val  = int(cm[i, j])
                cell = ws3.cell(row=3 + i, column=col_cursor + 1 + j, value=val)
                cell.alignment = _CENTER
                cell.border    = _THIN_BORDER
                if i == j:  # diagonal → green
                    cell.fill = PatternFill('solid', start_color='C6EFCE')
                    cell.font = Font(name='Arial', bold=True, color='276221')
                else:
                    cell.font = _BODY_FONT

        # Per-class F1 from classification report
        f1_row = 3 + n_cls + 1
        ws3.cell(row=f1_row, column=col_cursor,
                 value='Per-Class F1').font = Font(bold=True, name='Arial')
        for j in range(n_cls):
            key = str(j)
            f1  = rep.get(key, {}).get('f1-score', '')
            ws3.cell(row=f1_row, column=col_cursor + 1 + j,
                     value=round(f1, 4) if f1 != '' else '').font = _BODY_FONT

        col_cursor += n_cls + 3   # gap between model tables

    # ── Sheet 4: Plots ─────────────────────────────────────────────────────────
    ws4 = wb.create_sheet('Plots')
    title_cell = ws4.cell(row=1, column=1, value='CREDIT — All Output Plots')
    title_cell.font = Font(name='Arial', bold=True, size=14)

    img_row = 3
    for label, img_path in plot_paths:
        if not Path(img_path).exists():
            continue
        ws4.cell(row=img_row, column=1,
                 value=label).font = Font(name='Arial', bold=True, size=11)
        img_row += 1

        xl_img = XLImage(img_path)
        orig_w, orig_h = xl_img.width, xl_img.height
        target_w = 900
        scale         = target_w / orig_w if orig_w > 0 else 1
        xl_img.width  = int(orig_w * scale)
        xl_img.height = int(orig_h * scale)

        cell_addr = f'A{img_row}'
        ws4.add_image(xl_img, cell_addr)
        # openpyxl row height is in points; 1pt ≈ 0.75px
        ws4.row_dimensions[img_row].height = xl_img.height * 0.75
        img_row += int(xl_img.height / 15) + 3

        # ── Extra sheets: per-pixel AU / EU / TU + threshold analysis ──────────────
    if pixel_uncertainty:
        add_pixel_uncertainty_sheets(wb, pixel_uncertainty)

    wb.save(xlsx_path)
    print(f'\n✅ Excel report saved → {xlsx_path}')
    return str(xlsx_path)

# 6.0 — CREDIT Distillation

Generates soft targets from the pre-trained ensemble teachers, builds a dual-head student for each architecture, and trains it via KL divergence (aleatoric head) + MSE (epistemic head).


## 6.1 — Soft Target Generation

Loads each ensemble teacher, runs inference over the training data, and derives CREDIT targets: `p_star` (normalised per-class minimum — aleatoric proxy) and `delta_p` (per-class prediction spread — epistemic proxy).


In [17]:
try:
    keras.config.enable_unsafe_deserialization()
except Exception:
    pass

def get_ensemble_paths(model_name):
    """Locate ensemble teacher weights using multiple fallback search patterns."""
    # Obfuscated name mapping (original→k0NN codes uploaded as dias-ensemble-models dataset)
    _OBF_MAP = {
        "AlexNet_CNN": ["k016", "k017", "k018", "k019", "k020"],
        "GFNet":       ["k021", "k022", "k023", "k024", "k025"],
        "ViT_UNet":    ["k026", "k027", "k028", "k029", "k030"],
    }

    paths = []

    # 1. Original names in ENSEMBLE_DIR
    if ENSEMBLE_DIR is not None:
        pattern = str(ENSEMBLE_DIR / f"{model_name}_ens_*_final.keras")
        paths = glob.glob(pattern)

    # 2. Obfuscated names in ENSEMBLE_DIR (dias-ensemble-models dataset)
    if not paths and ENSEMBLE_DIR is not None and model_name in _OBF_MAP:
        for code in _OBF_MAP[model_name]:
            p = ENSEMBLE_DIR / f"{code}.keras"
            if p.exists():
                paths.append(str(p))

    # 3. Recursive search for original names across all /kaggle/kin
    if not paths:
        paths = glob.glob(
            f"/kaggle/kin/**/{model_name}_ens_*_final.keras",
            recursive=True
        )

    # 4. Recursive search for obfuscated names across all /kaggle/kin
    if not paths and model_name in _OBF_MAP:
        for code in _OBF_MAP[model_name]:
            found = glob.glob(f"/kaggle/kin/**/{code}.keras", recursive=True)
            paths.extend(found)

    return sorted(paths)


def generate_credit_targets(ensemble_paths, x_data, batch_size=128):
    """
    Run each teacher over x_data and derive CREDIT targets (FIX H14).

    Returns a dict of targets for the student heads:
        p_star  : (N, C) ensemble mean probability (a proper distribution)
        p_low   : (N, C) per-class lower credal bound  = min over members
        delta_p : (N, C) credal interval width          = max - min over members
        au      : (N, 1) aleatoric part = mean member entropy / ln(C)   (in [0, 1])
    """
    all_preds = []
    for path in ensemble_paths:
        print(f'  Loading teacher: {path}')
        model = tf.keras.models.load_model(path, compile=False, safe_mode=False)
        all_preds.append(model.predict(x_data, batch_size=batch_size, verbose=1))
        del model
        tf.keras.backend.clear_session()

    stacked = tf.stack(all_preds, axis=0)   # (M, N, C)
    p_min   = tf.reduce_min(stacked, axis=0)  # (N, C)
    p_max   = tf.reduce_max(stacked, axis=0)  # (N, C)

    n_cls   = int(stacked.shape[-1])
    p_mean  = tf.reduce_mean(stacked, axis=0)
    ent_m   = -tf.reduce_sum(stacked * tf.math.log(tf.clip_by_value(stacked, 1e-12, 1.0)), axis=-1)
    au_t    = tf.reduce_mean(ent_m, axis=0, keepdims=False)[:, None] / np.log(n_cls)
    return {'p_star': p_mean, 'p_low': p_min, 'delta_p': p_max - p_min,
            'au': tf.clip_by_value(au_t, 0.0, 1.0)}


def credit_uncertainty(outputs, num_classes):
    """FIX H14: coherent CREDIT uncertainty from the student's four heads (all in [0, 1]).

    TU = H(p_star) / ln C ; AU = predicted mean member entropy (capped at TU) ; EU = TU - AU
    (mutual-information decomposition, so AU + EU = TU holds exactly).
    Credal bounds: lower = p_low, upper = min(1, p_low + delta_p).
    """
    p_star, p_low, delta_p, au_hat = [np.asarray(o, dtype=np.float64) for o in outputs]
    p_star = np.clip(p_star, 1e-12, 1.0)
    tu = -np.sum(p_star * np.log(p_star), axis=-1) / np.log(num_classes)
    au = np.minimum(au_hat.reshape(-1), tu)
    eu = tu - au
    lower = np.clip(p_low, 0.0, 1.0)
    upper = np.clip(lower + delta_p, 0.0, 1.0)
    return p_star, lower, upper, au, eu, tu

## 6.2 — CREDIT Student (Dual-Head)

Wraps any single-head base architecture with two output heads: a softmax head for the aleatoric belief (`p_star`) and a sigmoid head for the epistemic spread (`delta_p`).


In [18]:
def build_credit_student(base_builder_func, num_classes):
    """
    Attach CREDIT dual-output heads to any single-head base architecture.

    Args:
        base_builder_func: Zero-argument callable that returns a Keras model.
        num_classes:       Number of target classes.

    Returns:
        A Keras model with outputs [p_star (softmax), delta_p (sigmoid)].
    """
    base_model = base_builder_func()
    features   = base_model.layers[-2].output   # penultimate feature tensor

    p_star  = layers.Dense(num_classes, activation='softmax',  name='p_star' )(features)
    p_low   = layers.Dense(num_classes, activation='sigmoid',  name='p_low'  )(features)   # FIX H14
    delta_p = layers.Dense(num_classes, activation='sigmoid',  name='delta_p')(features)
    au      = layers.Dense(1,           activation='sigmoid',  name='au'     )(features)   # FIX H14

    return tf.keras.Model(
        inputs=base_model.input,
        outputs={'p_star': p_star, 'p_low': p_low, 'delta_p': delta_p, 'au': au},
        name='CREDIT_Student'
    )

## 6.3 — Model Builder Registry

Centralised registry mapping model names to zero-argument builder lambdas.


In [19]:
model_builders = {
    'AlexNet_CNN': lambda: build_alexnet(
        input_shape, num_classes,
        dropout_rate=DROPOUT_RATE, cfg=ALEXNET_CFG
    ),
    'GFNet': lambda: build_gfnet(
        input_shape, num_classes,
        inner_patch=INNER_PATCH,
        hidden_dim=GFNET_CFG['hidden_dim'],
        num_blocks=GFNET_CFG['num_blocks'],
        mlp_ratio=GFNET_CFG['mlp_ratio'],
        dropout_rate=DROPOUT_RATE
    ),
    'ViT_UNet': lambda: build_vit_unet_singlehead(
        input_shape, num_classes,
        inner_patch=INNER_PATCH,
        projection_dim=VIT_CFG['projection_dim'],
        num_heads=VIT_CFG['num_heads'],
        transformer_layers=VIT_CFG['transformer_layers'],
        mlp_multiplier=VIT_CFG['mlp_multiplier'],
        dropout_rate=DROPOUT_RATE,
        head_units=tuple(VIT_CFG['head_units'])
    ),
}

## 6.4 — Verify Ensemble Teachers Available


In [20]:
# Debug: list what's available in kaggle input
import subprocess
result = subprocess.run(['find', '/kaggle/kin', '-maxdepth', '4', '-name', '*.keras', '-o', '-name', '*.mat'], capture_output=True, text=True)
print('Input files found:')
print(result.stdout[:2000])
print('---')

# Verify all ensemble teacher models are accessible before training
print('=== Verifying ensemble teacher model availability ===')
all_found = True
for mname in model_builders:
    paths = get_ensemble_paths(mname)
    status = '✅' if len(paths) == 5 else ('⚠️' if paths else '❌')
    print(f'  {status} {mname}: {len(paths)} models found')
    for p in paths:
        sz = Path(p).stat().st_size / 1024 / 1024
        print(f'      {Path(p).name} ({sz:.1f} MB)')
    if len(paths) == 0:
        all_found = False

if not all_found:
    print('\n⚠️  Some ensemble models missing – listing all .keras files in input:')
    import subprocess
    result = subprocess.run(['find', '/kaggle/kin/dias-ensemble-k13', '-name', '*.keras'],
                           capture_output=True, text=True)
    print(result.stdout[:3000])


Input files found:
/kaggle/kin/datasets/daneshselwal/ip-dataset-k01/Indian_pines.mat
/kaggle/kin/datasets/daneshselwal/ip-dataset-k01/k001.mat
/kaggle/kin/datasets/daneshselwal/ip-dataset-k01/Indian_pines_gt.mat
/kaggle/kin/datasets/daneshselwal/ip-dataset-k01/k002.mat

---
=== Verifying ensemble teacher model availability ===
  ✅ AlexNet_CNN: 5 models found
      AlexNet_CNN_ens_1_final.keras (260.9 MB)
      AlexNet_CNN_ens_2_final.keras (260.9 MB)
      AlexNet_CNN_ens_3_final.keras (260.9 MB)
      AlexNet_CNN_ens_4_final.keras (260.9 MB)
      AlexNet_CNN_ens_5_final.keras (260.9 MB)
  ✅ GFNet: 5 models found
      GFNet_ens_1_final.keras (132.8 MB)
      GFNet_ens_2_final.keras (132.8 MB)
      GFNet_ens_3_final.keras (132.8 MB)
      GFNet_ens_4_final.keras (132.8 MB)
      GFNet_ens_5_final.keras (132.8 MB)
  ✅ ViT_UNet: 5 models found
      ViT_UNet_ens_1_final.keras (190.7 MB)
      ViT_UNet_ens_2_final.keras (190.7 MB)
      ViT_UNet_ens_3_final.keras (190.7 MB)
      ViT_UN

## 6.5 — Training Loop

For each architecture: locate ensemble teachers → generate soft targets → build `tf.data` pipelines → build and compile the CREDIT student → train with `ModelCheckpoint` → compute a quick post-training uncertainty summary.


In [21]:
train_results = []

for model_name, builder in model_builders.items():
    print(f"\n{'='*25} CREDIT Distillation: {model_name} {'='*25}")

    # ── 1. Locate ensemble teachers ────────────────────────────────────────────
    ensemble_paths = get_ensemble_paths(model_name)
    if len(ensemble_paths) == 0:
        print('  No ensemble models found — skipping.')
        continue
    if len(ensemble_paths) < 5:
        print(f'  Warning: only {len(ensemble_paths)} ensemble models found (expected 5).')

    # ── 2. Select correct data split ───────────────────────────────────────────
    # FIX C3: the student is checkpointed on the validation split, the test split stays unseen
    x_tr, x_va, x_te = x_train, x_val, x_test

    # ── 3. Generate soft targets ───────────────────────────────────────────────
    print('  Generating training targets...')
    tgt_tr = generate_credit_targets(ensemble_paths, x_tr)

    print('  Generating validation targets...')
    tgt_va = generate_credit_targets(ensemble_paths, x_va)

    # ── 4. Build tf.data pipelines ─────────────────────────────────────────────
    train_ds = (
        tf.data.Dataset
        .from_tensor_slices((x_tr, tgt_tr))
        .shuffle(1024).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    )
    test_ds = (
        tf.data.Dataset
        .from_tensor_slices((x_va, tgt_va))
        .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    )

    # ── 5. Build & compile student ─────────────────────────────────────────────
    tf.keras.backend.clear_session()
    student = build_credit_student(builder, num_classes)

    optimizer = (
        keras.optimizers.Adagrad(learning_rate=ALEXNET_LR_START)
        if model_name == 'AlexNet_CNN'
        else keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    )

    student.compile(
        optimizer=optimizer,
        loss={
            'p_star':  tf.keras.losses.KLDivergence(),
            'p_low':   tf.keras.losses.MeanSquaredError(),
            'delta_p': tf.keras.losses.MeanSquaredError(),
            'au':      tf.keras.losses.MeanSquaredError(),
        },
        loss_weights={'p_star': 1.0, 'p_low': 0.5, 'delta_p': 0.5, 'au': 0.5},   # lambda = 0.5
    )

    # ── 6. Train ───────────────────────────────────────────────────────────────
    best_path = MODEL_DIR / f'{model_name}_CREDIT_best.keras'
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            filepath=str(best_path), monitor='val_loss',
            mode='min', save_best_only=True, verbose=1
        )
    ]

    t0 = time.perf_counter()
    student.fit(
        train_ds, validation_data=test_ds,
        epochs=EPOCHS, callbacks=callbacks, verbose=1
    )
    train_time = time.perf_counter() - t0

    # ── 7. Post-training uncertainty summary ───────────────────────────────────
    student.load_weights(best_path)
    _o = student.predict(x_va, batch_size=BATCH_SIZE)
    _, _, _, au, eu, tu = credit_uncertainty([_o['p_star'], _o['p_low'], _o['delta_p'], _o['au']], num_classes)

    train_results.append({
        'Model':          model_name,
        'Mean_AU':        np.mean(au),
        'Mean_EU':        np.mean(eu),
        'Mean_TU':        np.mean(tu),
        'Train_Time_sec': train_time,
    })

# ── Save training uncertainty summary ─────────────────────────────────────────
summary_df   = pd.DataFrame(train_results)
summary_path = RESULTS_DIR / 'credit_train_uncertainty_summary.csv'
summary_df.to_csv(summary_path, index=False)
print(f'\nSaved training summary → {summary_path}')
print(summary_df)

# Stash for the final Excel report
_credit_train_summary_df = summary_df.copy()


========================= CREDIT Distillation: AlexNet_CNN =========================
  Generating training targets...
  Loading teacher: /kaggle/working/models/ensembles/AlexNet_CNN_ens_1_final.keras


 1/24 ━━━━━━━━━━━━━━━━━━━━ 53s 2s/step

 4/24 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step

 7/24 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step

10/24 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step

13/24 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step

... (8,351 outputs left out)

7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 666ms/step

8/8 ━━━━━━━━━━━━━━━━━━━━ 9s 670ms/step



Saved training summary → /kaggle/working/results/credit_train_uncertainty_summary.csv
         Model   Mean_AU   Mean_EU   Mean_TU  Train_Time_sec
0  AlexNet_CNN  0.105005  0.164643  0.269648      201.928748
1        GFNet  0.002226  0.028250  0.030475      272.841806
2     ViT_UNet  0.004414  0.020616  0.025030      489.141164


# 7.0 — Evaluation

Loads the best saved CREDIT student weights for each architecture and computes a full suite of classification and calibration metrics — no re-training.


## 7.1 — Load, Predict & Compute Metrics


In [22]:
eval_results     = []
credit_artifacts = {}
credal_records   = []   # (model_name, credal_set_metrics dict) for the coverage sheet

for model_name, builder in model_builders.items():
    best_path = MODEL_DIR / f'{model_name}_CREDIT_best.keras'

    if not best_path.exists():
        print(f'  [{model_name}] No saved weights at {best_path} — skipping.')
        continue

    print(f"\n{'='*25} Evaluating: {model_name} {'='*25}")
    tf.keras.backend.clear_session()

    student = build_credit_student(builder, num_classes)
    student.load_weights(best_path)

    # Select the correct test split for this architecture
    if model_name == 'AlexNet_CNN':
        x_te, y_te_true, y_te_cat = x_test_alex, y_test_alex, y_test_alex_cat
    else:
        x_te, y_te_true, y_te_cat = x_test, y_test, y_test_cat

    _o = student.predict(x_te, batch_size=BATCH_SIZE)
    p_star_pred, c_lower, c_upper, au, eu, tu = credit_uncertainty(
        [_o['p_star'], _o['p_low'], _o['delta_p'], _o['au']], num_classes)

    y_pred = np.argmax(p_star_pred, axis=-1)

    # FIX H14: credal prediction sets from proper bounds [p_low, p_low + delta_p]
    credal_m = credal_set_metrics(
        lower=c_lower, upper=c_upper,
        y_true=y_te_true, num_classes=num_classes
    )
    credal_records.append((model_name, credal_m))

    credit_artifacts[model_name] = {
        'confusion_matrix': confusion_matrix(y_te_true, y_pred, labels=np.arange(num_classes)),
        'report':           classification_report(y_te_true, y_pred, labels=np.arange(num_classes), output_dict=True, zero_division=0
        ),
    }

    eval_results.append({
        'Model':         model_name,
        'Test_Accuracy': float(accuracy_score(y_te_true, y_pred)),
        'Macro_F1':      float(f1_score(y_te_true, y_pred, average='macro')),
        'Cohen_Kappa':   float(cohen_kappa_score(y_te_true, y_pred)),
        'Test_NLL':      float(log_loss(y_te_true, p_star_pred,
                                        labels=np.arange(num_classes))),
        'Test_Brier':    float(multiclass_brier_score(y_te_cat, p_star_pred)),
        'Test_ECE':      float(expected_calibration_error(
            y_te_true, p_star_pred, n_bins=15)),
        'Mean_AU':       float(np.mean(au)),
        'Mean_EU':       float(np.mean(eu)),
        'Mean_TU':       float(np.mean(tu)),
        'Credal_Coverage':     credal_m['coverage'],
        'Credal_Avg_Set_Size': credal_m['avg_size'],
    })

eval_df = pd.DataFrame(eval_results)
eval_df.to_csv(RESULTS_DIR / 'credit_evaluation_summary.csv', index=False)
print('\n--- CREDIT Evaluation Summary ---')
print(eval_df.to_string(index=False))

# Credal coverage tables (overall + class-wise)
_credal_overall_df, _credal_classwise_df = credal_frames(credal_records)
_credal_overall_df.to_csv(RESULTS_DIR / 'credit_credal_coverage_overall.csv', index=False)
_credal_classwise_df.to_csv(RESULTS_DIR / 'credit_credal_coverage_classwise.csv', index=False)
print('\n--- CREDIT Credal Coverage (overall) ---')
print(_credal_overall_df.to_string(index=False))
print('\n--- CREDIT Credal Coverage (class-wise) ---')
print(_credal_classwise_df.to_string(index=False))

# Stash for the final Excel report
_credit_eval_df        = eval_df.copy()
_credit_artifacts_dict = credit_artifacts


========================= Evaluating: AlexNet_CNN =========================


 1/22 ━━━━━━━━━━━━━━━━━━━━ 15s 753ms/step

 3/22 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step  

 6/22 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step

 9/22 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step

12/22 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step

... (27 outputs left out)

22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 297ms/step

22/22 ━━━━━━━━━━━━━━━━━━━━ 11s 298ms/step



--- CREDIT Evaluation Summary ---
      Model  Test_Accuracy  Macro_F1  Cohen_Kappa  Test_NLL  Test_Brier  Test_ECE  Mean_AU  Mean_EU  Mean_TU  Credal_Coverage  Credal_Avg_Set_Size
AlexNet_CNN       0.439345  0.294808     0.347361  1.867445    0.773558  0.226099 0.126733 0.195304 0.322037         0.470651             1.964070
      GFNet       0.735681  0.626543     0.702870  1.648379    0.453809  0.199342 0.002341 0.064168 0.066509         0.729278             1.000000
   ViT_UNet       0.729278  0.581682     0.693139  1.626971    0.486909  0.224930 0.007415 0.050399 0.057814         0.638207             1.131982

--- CREDIT Credal Coverage (overall) ---
      Model  N_Test_Pixels  Overall_Empirical_Coverage  Average_Set_Size  Mean_Class_Coverage
AlexNet_CNN           2811                    0.470651          1.964070             0.261579
      GFNet           2811                    0.729278          1.000000             0.655116
   ViT_UNet           2811                    0.63820

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


## 7.2 — Confusion Matrices

Renders and saves a side-by-side confusion matrix heatmap for every evaluated model.


In [23]:
if credit_artifacts:
    class_ticks = [str(i + 1) for i in range(num_classes)]
    n_models    = len(credit_artifacts)
    fig, axes   = plt.subplots(1, n_models, figsize=(7 * n_models, 5.5))
    if n_models == 1:
        axes = [axes]

    for ax, (mname, art) in zip(axes, credit_artifacts.items()):
        sns.heatmap(
            art['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
            xticklabels=class_ticks, yticklabels=class_ticks, cbar=False, ax=ax
        )
        ax.set_title(f'{mname} — CREDIT\nConfusion Matrix', fontsize=13)
        ax.set_xlabel('Predicted Class')
        ax.set_ylabel('True Class')

    plt.tight_layout()
    save_path = RESULTS_DIR / 'credit_confusion_matrices.png'
    plt.savefig(save_path, dpi=220, bbox_inches='tight')
    print(f'Saved → {save_path}')
    plt.close()

Saved → /kaggle/working/results/credit_confusion_matrices.png


# 8.0 — Spatial Uncertainty Mapping

Runs full-scene inference for each CREDIT student and produces a 3×4 figure showing: the base prediction map, binary certain/uncertain masks, grey-overlay maps, and per-class pixel-count bar charts — all for the aleatoric, epistemic, and total uncertainty channels.


## 8.1 — Extract Full-Scene Patch Array


In [24]:
print('Extracting full-scene patch array (all H×W pixels)...')
pad   = PATCH_SIZE // 2
x_pad = np.pad(x_img, ((pad, pad), (pad, pad), (0, 0)), mode='edge')

scene_patches = np.empty((H * W, PATCH_SIZE, PATCH_SIZE, B), dtype=np.float32)
for idx, (r, c) in enumerate(
    [(r, c) for r in range(H) for c in range(W)]
):
    scene_patches[idx] = x_pad[r:r + PATCH_SIZE, c:c + PATCH_SIZE, :]

print(f'Scene patch tensor: {scene_patches.shape}')

Extracting full-scene patch array (all H×W pixels)...


Scene patch tensor: (21025, 9, 9, 220)


## 8.2 — Spatial Map Generator

Defines the function that produces and saves the 3×4 uncertainty figure for a given student model.


In [25]:
import colorsys

CLASS_COLOR_BASE = [
    '#0000FF', '#00FF00', '#FF0000', '#00FFFF',
    '#FF00FF', '#FFFF00', '#A52A2A', '#FFA500',
    '#7FFF00', '#8A2BE2',
]
UNCERTAIN_COLOR = '#808080'

def _generate_extra_shades(n_extra, hue_offset=0.05, sat=0.85, val=0.95):
    """Bold, highly-saturated extra hues in the same style as CLASS_COLOR_BASE."""
    shades = []
    for i in range(n_extra):
        hue = (hue_offset + i / max(n_extra, 1)) % 1.0
        r, g, b = colorsys.hsv_to_rgb(hue, sat, val)
        shades.append('#%02x%02x%02x' % (int(r * 255), int(g * 255), int(b * 255)))
    return shades

def get_class_colors(n):
    """Return n distinct hex colours, extending the base 10 with additional bold
    shades in the same saturated style if needed."""
    if n <= len(CLASS_COLOR_BASE):
        return CLASS_COLOR_BASE[:n]
    return CLASS_COLOR_BASE + _generate_extra_shades(n - len(CLASS_COLOR_BASE))

def resolve_uncertain_color(class_colors, preferred=UNCERTAIN_COLOR, min_dist=60):
    """Return `preferred` unless it visually clashes with any class colour, in
    which case fall back to a distinct dark slate so 'Uncertain' never collides
    with a real class colour."""
    def _hex_to_rgb(h):
        h = h.lstrip('#')
        return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))
    pr, pg, pb = _hex_to_rgb(preferred)
    for c in class_colors:
        cr, cg, cb = _hex_to_rgb(c)
        dist = ((pr - cr) ** 2 + (pg - cg) ** 2 + (pb - cb) ** 2) ** 0.5
        if dist < min_dist:
            return '#2F2F2F'
    return preferred


def generate_spatial_credit_maps(
    model_name, student_model, scene_pixels,
    H, W, au_thresh=0.5, eu_thresh=0.2, tu_thresh=0.7
):
    """
    Run full-scene inference and save a 3×4 spatial uncertainty figure.

    Panel layout (rows × cols):
      Row 0: Base prediction | AU binary | EU binary | TU binary
      Row 1: AU grey overlay | EU grey overlay | TU grey overlay | (blank)
      Row 2: AU bar chart   | EU bar chart   | TU bar chart   | (blank)
    """
    print(f'  Inferring full scene for {model_name}...')
    _o = student_model.predict(scene_pixels, batch_size=2048, verbose=1)
    n_cls = _o['p_star'].shape[-1]
    # FIX H14 / M22: normalised AU / EU / TU in [0, 1] (comparable across datasets)
    p_star_scene, _, _, au_scene, eu_scene, tu_scene = credit_uncertainty(
        [_o['p_star'], _o['p_low'], _o['delta_p'], _o['au']], n_cls)

    pred_map = np.argmax(p_star_scene, axis=-1).reshape(H, W)
    au_mask  = (au_scene.reshape(H, W) > au_thresh).astype(int)
    eu_mask  = (eu_scene.reshape(H, W) > eu_thresh).astype(int)
    tu_mask  = (tu_scene.reshape(H, W) > tu_thresh).astype(int)

    combined_au = np.where(au_mask == 1, n_cls, pred_map)
    combined_eu = np.where(eu_mask == 1, n_cls, pred_map)
    combined_tu = np.where(tu_mask == 1, n_cls, pred_map)

    class_colors = get_class_colors(n_cls)
    uncertain_color = resolve_uncertain_color(class_colors)
    cmap_base   = ListedColormap(class_colors)
    cmap_unc    = ListedColormap(class_colors + [uncertain_color])
    cmap_binary = ListedColormap(['#FFFF00', '#001F3F'])

    bar_lbls = [f'Class {i}' for i in range(n_cls)] + ['Uncertain']
    bar_cols = class_colors + [uncertain_color]

    fig, axes = plt.subplots(3, 4, figsize=(38, 26))
    fig.suptitle(
        f'{model_name} — CREDIT Uncertainty Maps (Absolute Thresholds)',
        fontsize=24, fontweight='bold', y=0.99
    )

    # ── Row 0: Base prediction + 3 binary maps ─────────────────────────────────
    axes[0, 0].imshow(pred_map, cmap=cmap_base, vmin=0, vmax=n_cls - 1)
    axes[0, 0].set_title('Base Prediction Map', fontsize=15)
    axes[0, 0].axis('off')

    binary_specs = [
        (axes[0, 1], au_mask, f'Aleatoric (AU > {au_thresh})'),
        (axes[0, 2], eu_mask, f'Epistemic (EU > {eu_thresh})'),
        (axes[0, 3], tu_mask, f'Total     (TU > {tu_thresh})'),
    ]
    for ax, mask, label in binary_specs:
        ax.imshow(mask, cmap=cmap_binary, vmin=0, vmax=1)
        ax.set_title(f'Certain vs Uncertain\n{label}', fontsize=15, pad=10)
        ax.axis('off')
        ax.legend(
            handles=[
                Patch(facecolor='#FFFF00', label='Certain'),
                Patch(facecolor='#001F3F', label='Uncertain'),
            ],
            loc='upper left',
            bbox_to_anchor=(0.0, -0.02),
            borderaxespad=0,
            fontsize=11,
            framealpha=0.9,
            ncol=2
        )

    # ── Row 1: Grey overlay maps ───────────────────────────────────────────────
    overlay_specs = [
        (axes[1, 0], combined_au, f'Aleatoric (AU > {au_thresh})'),
        (axes[1, 1], combined_eu, f'Epistemic (EU > {eu_thresh})'),
        (axes[1, 2], combined_tu, f'Total     (TU > {tu_thresh})'),
    ]
    for ax, combined, label in overlay_specs:
        ax.imshow(combined, cmap=cmap_unc, vmin=0, vmax=n_cls)
        ax.set_title(f'Grey Overlay — {label}', fontsize=15, pad=10)
        ax.axis('off')
    axes[1, 3].axis('off')

    # ── Row 2: Pixel-count bar charts ──────────────────────────────────────────
    bar_specs = [
        (axes[2, 0], combined_au, f'Aleatoric (AU > {au_thresh})'),
        (axes[2, 1], combined_eu, f'Epistemic (EU > {eu_thresh})'),
        (axes[2, 2], combined_tu, f'Total     (TU > {tu_thresh})'),
    ]
    for ax, combined, label in bar_specs:
        uniq, cnt = np.unique(combined, return_counts=True)
        c_dict    = {int(k): int(v) for k, v in zip(uniq, cnt)}
        bar_vals  = [c_dict.get(i, 0) for i in range(n_cls + 1)]
        ax.bar(bar_lbls, bar_vals, color=bar_cols, edgecolor='black')
        ax.set_title(f'Pixel Counts — {label}', fontsize=15, pad=10)
        ax.tick_params(axis='x', rotation=45, labelsize=11)
        ax.set_ylabel('Pixel Count', fontsize=12)
        max_val = max(bar_vals, default=1)
        for i, v in enumerate(bar_vals):
            ax.text(i, v + max_val * 0.01,
                    f'{v:,}', ha='center', va='bottom',
                    fontweight='bold', fontsize=9)
        ax.set_ylim(0, max_val * 1.12)
    axes[2, 3].axis('off')

    plt.tight_layout(rect=[0, 0.03, 1, 0.98])
    save_path = RESULTS_DIR / f'{model_name}_CREDIT_spatial_maps.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f'  Saved → {save_path}')
    plt.close()
    
    # Return per-pixel values so they can be exported to Excel (section 9.0)
    return {
        'au': au_scene, 'eu': eu_scene, 'tu': tu_scene,
        'pred': pred_map.reshape(-1),          # 0-based class index per pixel (row-major)
        'H': H, 'W': W, 'n_classes': n_cls,
        'thresholds': (au_thresh, eu_thresh, tu_thresh),
    }

In [26]:
# ── Pixel-level uncertainty table + threshold analysis (for the Excel report) ──
EXCEL_MAX_DATA_ROWS    = 1_048_575            # Excel limit (1,048,576 rows) minus the header row
SENSITIVITY_THRESHOLDS = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0]


def build_pixel_uncertainty_df(unc, y_ref):
    """
    One row per pixel that is uncertain under AU, EU or TU (union of the three masks).
    Row / Col / Pixel_ID are 1-based (Pixel_ID = (Row-1)*W + Col, row-major).
    Predicted_Class is 1-based. Reference_Class is the .mat label (0 = unlabelled).
    Sorted by TU, highest first.
    """
    au, eu, tu = unc['au'], unc['eu'], unc['tu']
    W = unc['W']
    au_t, eu_t, tu_t = unc['thresholds']

    f_au, f_eu, f_tu = au > au_t, eu > eu_t, tu > tu_t
    idx = np.flatnonzero(f_au | f_eu | f_tu)

    r0, c0 = np.divmod(idx, W)
    tags = np.char.add(
        np.char.add(np.where(f_au[idx], 'AU ', ''), np.where(f_eu[idx], 'EU ', '')),
        np.where(f_tu[idx], 'TU', '')
    )
    under = pd.Series(tags).str.strip().str.replace(' ', '+', regex=False)

    df = pd.DataFrame({
        'Pixel_ID':        idx + 1,
        'Row':             r0 + 1,
        'Col':             c0 + 1,
        'Predicted_Class': unc['pred'][idx] + 1,
        'Reference_Class': np.asarray(y_ref).reshape(-1)[idx],
        'AU':              au[idx].astype(np.float64).round(6),
        'EU':              eu[idx].astype(np.float64).round(6),
        'TU':              tu[idx].astype(np.float64).round(6),
        f'AU_Uncertain (>{au_t})': f_au[idx].astype(int),
        f'EU_Uncertain (>{eu_t})': f_eu[idx].astype(int),
        f'TU_Uncertain (>{tu_t})': f_tu[idx].astype(int),
        'Uncertain_Under': under.values,
    })
    return df.sort_values('TU', ascending=False, kind='mergesort').reset_index(drop=True)


def build_threshold_analysis(unc):
    """Summary tables: is AU+EU=TU, and how much do the thresholds matter?"""
    au, eu, tu = unc['au'], unc['eu'], unc['tu']
    n = au.size
    au_t, eu_t, tu_t = unc['thresholds']
    f_au, f_eu, f_tu = au > au_t, eu > eu_t, tu > tu_t

    counts = pd.DataFrame({
        'Criterion':        ['AU', 'EU', 'TU'],
        'Threshold':        [au_t, eu_t, tu_t],
        'Uncertain_Pixels': [int(f_au.sum()), int(f_eu.sum()), int(f_tu.sum())],
    })
    counts['Percent_of_Scene'] = (100.0 * counts['Uncertain_Pixels'] / n).round(3)

    regions = [
        ('AU only', 1, 0, 0), ('EU only', 0, 1, 0), ('TU only', 0, 0, 1),
        ('AU & EU only', 1, 1, 0), ('AU & TU only', 1, 0, 1), ('EU & TU only', 0, 1, 1),
        ('AU & EU & TU', 1, 1, 1), ('None (certain under all three)', 0, 0, 0),
    ]
    overlap = pd.DataFrame([
        {'Region (mutually exclusive)': name,
         'Pixels': int(((f_au == bool(a)) & (f_eu == bool(e)) & (f_tu == bool(t))).sum())}
        for name, a, e, t in regions
    ])

    au_vs_tu = pd.DataFrame({
        'Comparison': ['AU-uncertain but NOT TU-uncertain', 'TU-uncertain but NOT AU-uncertain'],
        'Pixels':     [int((f_au & ~f_tu).sum()), int((f_tu & ~f_au).sum())],
    })

    diff = np.abs(au.astype(np.float64) + eu.astype(np.float64) - tu.astype(np.float64))
    identity = pd.DataFrame({
        'Metric': ['max |AU + EU - TU|', 'mean |AU + EU - TU|', 'min (TU - AU)  (= min EU, must be >= 0)',
                   'max possible AU = ln(#classes)', 'max observed AU', 'max observed EU', 'max observed TU'],
        'Value':  [float(diff.max()), float(diff.mean()), float((tu - au).min()),
                   float(np.log(unc['n_classes'])), float(au.max()), float(eu.max()), float(tu.max())],
    })

    sens = pd.DataFrame({'Threshold': SENSITIVITY_THRESHOLDS})
    for name, arr in (('AU', au), ('EU', eu), ('TU', tu)):
        sens[f'{name} > t (pixels)'] = [int((arr > t).sum()) for t in SENSITIVITY_THRESHOLDS]
    for name, arr in (('AU', au), ('EU', eu), ('TU', tu)):
        sens[f'{name} > t (%)'] = [round(100.0 * float((arr > t).mean()), 3) for t in SENSITIVITY_THRESHOLDS]

    return {'counts': counts, 'overlap': overlap, 'au_vs_tu': au_vs_tu,
            'identity': identity, 'sensitivity': sens, 'n_scene_pixels': int(n)}


def prepare_pixel_uncertainty(model_name, unc, y_ref, results_dir):
    """Build the uncertain-pixel table (+ full CSV copy) and threshold analysis for one model."""
    df_full  = build_pixel_uncertainty_df(unc, y_ref)
    csv_path = Path(results_dir) / f'{model_name}_uncertain_pixels.csv'
    df_full.to_csv(csv_path, index=False)

    truncated = len(df_full) > EXCEL_MAX_DATA_ROWS
    df_xl     = df_full.iloc[:EXCEL_MAX_DATA_ROWS] if truncated else df_full
    print(f'  [{model_name}] uncertain pixels (AU|EU|TU): {len(df_full):,}  → full table: {csv_path}')
    if truncated:
        print(f'  ⚠️ [{model_name}] exceeds Excel row limit; sheet holds the top {len(df_xl):,} by TU. '
              f'Use the CSV for the rest.')
    return {
        'df': df_xl, 'n_uncertain_total': len(df_full), 'truncated': truncated,
        'csv_path': str(csv_path), 'analysis': build_threshold_analysis(unc),
        'thresholds': unc['thresholds'],
    }


def add_pixel_uncertainty_sheets(wb, pixel_uncertainty):
    """Add one 'Uncertain Pixels - <model>' sheet per model + one 'Threshold Analysis' sheet."""
    for mname, res in pixel_uncertainty.items():
        ws = wb.create_sheet(f'Uncertain Pixels - {mname}'[:31])
        df = res['df']
        ws.append(list(df.columns))
        _style_header_row(ws, 1, 1, len(df.columns))
        ws.row_dimensions[1].height = 32
        for row in df.itertuples(index=False, name=None):
            ws.append(row)
        for j, col in enumerate(df.columns, start=1):
            ws.column_dimensions[get_column_letter(j)].width = max(12, min(len(str(col)) + 4, 26))
        ws.freeze_panes = 'A2'
        ws.auto_filter.ref = f'A1:{get_column_letter(len(df.columns))}{max(len(df) + 1, 2)}'

    wa = wb.create_sheet('Threshold Analysis')
    wa.cell(row=1, column=1, value='CREDIT — Uncertain-pixel analysis (AU / EU / TU)').font = Font(
        name='Arial', bold=True, size=14)

    first = next(iter(pixel_uncertainty.values()))
    au_t, eu_t, tu_t = first['thresholds']
    notes = [
        f'Thresholds used: AU > {au_t}, EU > {eu_t}, TU > {tu_t}.',
        'TU is computed as TU = AU + EU in this pipeline, so the identity holds for every pixel (see "Identity check").',
        'Because EU >= 0, TU >= AU for every pixel. Pixel COUNTS can still differ per criterion because each uses its own threshold.',
        'AU is an entropy in nats (max = ln #classes); EU is a mean of sigmoid outputs in [0, 1]. Equal thresholds are not directly comparable.',
        'Row / Col / Pixel_ID are 1-based. Reference_Class 0 = unlabelled pixel. Pixel sheets list pixels uncertain under AU, EU or TU (sorted by TU).',
    ]
    r = 3
    for line in notes:
        wa.cell(row=r, column=1, value=line).font = _BODY_FONT
        r += 1
    r += 1

    for mname, res in pixel_uncertainty.items():
        a = res['analysis']
        wa.cell(row=r, column=1, value=f'{mname}').font = Font(name='Arial', bold=True, size=13, color='1F4E79')
        r += 1
        wa.cell(row=r, column=1,
                value=f"Scene pixels: {a['n_scene_pixels']:,}   |   Uncertain under AU, EU or TU: {res['n_uncertain_total']:,}"
                      + ('   |   ⚠️ pixel sheet truncated to Excel row limit (see CSV)' if res['truncated'] else '')
                ).font = _BODY_FONT
        r += 2
        r = _write_df_to_sheet(wa, a['counts'],      start_row=r, title='Pixels flagged at the chosen thresholds') + 1
        r = _write_df_to_sheet(wa, a['overlap'],     start_row=r, title='Overlap between AU, EU and TU masks') + 1
        r = _write_df_to_sheet(wa, a['au_vs_tu'],    start_row=r, title='AU vs TU') + 1
        r = _write_df_to_sheet(wa, a['identity'],    start_row=r, title='Identity check  AU + EU = TU') + 1
        r = _write_df_to_sheet(wa, a['sensitivity'], start_row=r, title='Threshold sensitivity (same threshold t applied to AU, EU and TU)') + 2

    wa.column_dimensions['A'].width = 44
    for col in 'BCDEFGHIJ':
        wa.column_dimensions[col].width = 20

## 8.3 — Run Spatial Mapping for All Models


In [27]:
scene_uncertainty = {}   # per-model AU / EU / TU arrays for the Excel pixel sheet

for model_name, builder in model_builders.items():
    best_path = MODEL_DIR / f'{model_name}_CREDIT_best.keras'
    if not best_path.exists():
        print(f'[{model_name}] No weights found — skipping spatial maps.')
        continue

    tf.keras.backend.clear_session()
    student = build_credit_student(builder, num_classes)
    student.load_weights(best_path)

    scene_uncertainty[model_name] = generate_spatial_credit_maps(
        model_name, student, scene_patches, H, W,
        au_thresh=0.5, eu_thresh=0.2, tu_thresh=0.7
    )

  Inferring full scene for AlexNet_CNN...


 1/11 ━━━━━━━━━━━━━━━━━━━━ 1:20 8s/step

 2/11 ━━━━━━━━━━━━━━━━━━━━ 3s 345ms/step

 3/11 ━━━━━━━━━━━━━━━━━━━━ 2s 308ms/step

 4/11 ━━━━━━━━━━━━━━━━━━━━ 2s 313ms/step

 5/11 ━━━━━━━━━━━━━━━━━━━━ 1s 314ms/step

... (32 outputs left out)

10/11 ━━━━━━━━━━━━━━━━━━━━ 0s 342ms/step

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 923ms/step

11/11 ━━━━━━━━━━━━━━━━━━━━ 17s 926ms/step


  Saved → /kaggle/working/results/ViT_UNet_CREDIT_spatial_maps.png


# 9.0 — Excel Report Export

Collects all saved plot paths (confusion matrices, spatial uncertainty maps, and any training-curve PNGs) and writes the final `CREDIT_Results.xlsx` workbook.


In [28]:
# ── Collect all plot paths ─────────────────────────────────────────────────────
_plot_entries = []

# Confusion matrix image
_cm_path = str(RESULTS_DIR / 'credit_confusion_matrices.png')
_plot_entries.append(('Confusion Matrices (all models)', _cm_path))

# Per-model spatial uncertainty maps
for _mname in model_builders:
    _sp_path = str(RESULTS_DIR / f'{_mname}_CREDIT_spatial_maps.png')
    _plot_entries.append((f'{_mname} — Spatial Uncertainty Maps', _sp_path))

# Any additional training-curve PNGs saved to PLOT_DIR
for _p in sorted(PLOT_DIR.glob('*.png')):
    _plot_entries.append((_p.stem.replace('_', ' ').title(), str(_p)))

# ── Pixel-level AU / EU / TU tables + threshold analysis (one sheet per model) ──
print('Building uncertain-pixel tables...')
_pixel_uncertainty = {
    _m: prepare_pixel_uncertainty(_m, _unc, y_img, RESULTS_DIR)
    for _m, _unc in scene_uncertainty.items()
}

# ── Write the workbook ─────────────────────────────────────────────────────────
create_credit_excel_report(
    results_dir           = RESULTS_DIR,
    train_summary_df      = _credit_train_summary_df,
    eval_summary_df       = _credit_eval_df,
    credit_artifacts_dict = _credit_artifacts_dict,
    plot_paths            = _plot_entries,
    pixel_uncertainty     = _pixel_uncertainty,
    credal_overall_df     = _credal_overall_df,
    credal_classwise_df   = _credal_classwise_df,
)

Building uncertain-pixel tables...
  [AlexNet_CNN] uncertain pixels (AU|EU|TU): 9,198  → full table: /kaggle/working/results/AlexNet_CNN_uncertain_pixels.csv
  [GFNet] uncertain pixels (AU|EU|TU): 3,603  → full table: /kaggle/working/results/GFNet_uncertain_pixels.csv
  [ViT_UNet] uncertain pixels (AU|EU|TU): 1,838  → full table: /kaggle/working/results/ViT_UNet_uncertain_pixels.csv



✅ Excel report saved → /kaggle/working/results/CREDIT_Results.xlsx


'/kaggle/working/results/CREDIT_Results.xlsx'

In [29]:
# ── Final output manifest ──────────────────────────────────────────────────────
import subprocess
print('\n=== CREDIT Output Files ===')
result = subprocess.run(
    ['find', '/kaggle/working/credit', '-type', 'f'],
    capture_output=True, text=True
)
for f in sorted(result.stdout.strip().split('\n')):
    if f:
        sz = Path(f).stat().st_size / 1024 / 1024
        print(f'  {f}  ({sz:.2f} MB)')
print('\n✅ CREDIT pipeline complete!')


=== CREDIT Output Files ===

✅ CREDIT pipeline complete!
